# Visualizing Neo4j Graph Data Science (GDS) Graphs

In [ ]:
%pip install graphdatascience
%pip install matplotlib
%pip install python-dotenv

In [ ]:
import dotenv

dotenv.load_dotenv()

## Setup GDS graph

To use GDS, you can either use GDS as a plugin or Aura Graph Analytics.
In the following, you can choose:

 * Provide Aura API credentials and and use Aura Graph Analytics.
 * Use Neo4j + GDS Plugin.

For more information, see the [GDS documentation](https://neo4j.com/docs/graph-data-science/current/installation/).

In [ ]:
import os

from graphdatascience.session import (
    GdsSessions,
    DbmsConnectionInfo,
    AuraAPICredentials,
    SessionMemory,
)
from graphdatascience import GraphDataScience

# Get Neo4j DB URI, credentials and name from environment if applicable
db_connection = DbmsConnectionInfo(
    aura_instance_id=os.environ.get("AURA_INSTANCEID"),
    username=os.environ.get("NEO4J_USERNAME", "neo4j"),
    password=os.environ.get("NEO4J_PASSWORD"),
    uri=os.environ["NEO4J_URI"],
)

session_name = "neo4j-viz-gds-example"
if os.environ.get("AURA_API_CLIENT_ID"):
    # Use Aura Graph Analytics
    sessions = GdsSessions(
        api_credentials=AuraAPICredentials(
            client_id=os.environ["AURA_API_CLIENT_ID"],
            client_secret=os.environ["AURA_API_CLIENT_SECRET"],
            project_id=os.environ.get("AURA_API_PROJECT_ID"),
        )
    )
    gds = sessions.get_or_create(
        session_name=session_name,
        memory=SessionMemory.m_2GB,
        db_connection=db_connection,
    )
else:
    # Use GDS Plugin
    sessions = None
    gds = GraphDataScience(
        endpoint=db_connection.get_uri(),
        auth=(db_connection.username, db_connection.password),
    )

In [ ]:
G = gds.graph.load_cora(graph_name="cora")

In [ ]:
# Run some algorithms to use later for visualization
gds.nodeSimilarity.mutate(
    G, mutateRelationshipType="SIMILAR", mutateProperty="similarity"
)
gds.pageRank.mutate(G, mutateProperty="pagerank")
gds.louvain.mutate(G, mutateProperty="componentId")

## Visualization

In [ ]:
from neo4j_viz.gds import from_gds

VG = from_gds(
    gds,
    G,
    max_node_count=100,
)

In [ ]:
VG.render()

### Changing captions

We can also change the node captions, if we want to see something other that the node labels.
For this dataset it might make sense to caption by scientific subject.

In [ ]:
for node in VG.nodes:
    node.caption = str(node.properties["subject"])

In [ ]:
VG.render()

## Sizing the nodes

Next, we can size the nodes by their pageRank score to show their importance.

In [ ]:
VG.resize_nodes(property="pagerank")
VG.render()

### Coloring

There are two main ways of coloring the nodes of a graph:

* By discrete color space, in which case a new colors will be given to each unique node field or property
* By continuous color space, in which case nodes will be colored according to a range of colors, according to their field or property value

We will start be coloring the node based on our discrete node property "subject" using the default colors.

In [ ]:
VG.color_nodes(property="subject")
VG.render()

Now, let us color by our continuous node field "size" that we computed above with PageRank, again using the default colors.
We set `override=True` so as to replace the previous coloring completely.
Note how the nodes are colored from yellow to purple, and how that also corresponds to the nodes' sizes.

In [ ]:
from neo4j_viz.colors import ColorSpace

VG.color_nodes(field="size", color_space=ColorSpace.CONTINUOUS, override=True)
VG.render()

#### Custom coloring

In some cases, the default colors are too few.
For example, if you have many communities, you might need a lot more colors.


In [ ]:
%pip install matplotlib, palettable

In [ ]:
from palettable.colorbrewer.qualitative import Dark2_7
import matplotlib.colors as mcolors

number_of_components = len({n.properties["componentId"] for n in VG.nodes})
print(f"Number of components: {number_of_components}")

linear_color_map = Dark2_7.mpl_colormap.resampled(number_of_components)
colors = [mcolors.rgb2hex(linear_color_map(i)) for i in range(number_of_components)]

VG.color_nodes(property="componentId", colors=colors, override=True)

In [ ]:
VG.render()

### Render options

Besides changing the appearance of our nodes and relationships, we can also modify the rendering itself.
For example, some available options are:

* graph layout
* renderer
* zoom level
* initial position 

In [ ]:
from neo4j_viz import Layout

VG.render(layout=Layout.CIRCULAR, initial_zoom=0.5)

## Saving the visualization

In [ ]:
from neo4j_viz.options import Renderer

os.makedirs("./out", exist_ok=True)

# Save the visualization to a file
with open("out/cora.html", "w") as f:
    f.write(VG.render(renderer=Renderer.CANVAS).data)

## Cleanup

Lets cleanup the graphs we created in GDS.

In [ ]:
gds.graph.drop("cora")

In [ ]:
if sessions:
    sessions.delete(session_name=session_name)